In [1]:
import optuna
DB_URL = "sqlite:///./saved_models/light/00_hyper/gat/study.db"
STUDY_NAME = "lightning_gat" # Ensure this matches your script
study = optuna.load_study(study_name=STUDY_NAME, storage=DB_URL)
# Print study statistics
print("Number of finished trials: ", len(study.trials))
trial = study.best_trial
print("Best trial:", trial)


Number of finished trials:  705
Best trial: FrozenTrial(number=698, state=<TrialState.COMPLETE: 1>, values=[2.230292558670044], datetime_start=datetime.datetime(2026, 7, 5, 8, 22, 58, 799331), datetime_complete=datetime.datetime(2026, 7, 5, 8, 55, 30, 110035), params={'lr': 9.302603517798478e-05, 'weight_decay': 4.005427972065182e-07, 'num_gat_layers': 7, 'gat_out_channels_0': 128, 'gat_heads_0': 8, 'gat_dropout_0': 0.2576869550803659, 'gat_out_channels_1': 32, 'gat_heads_1': 4, 'gat_dropout_1': 0.07584116081875439, 'gat_out_channels_2': 64, 'gat_heads_2': 8, 'gat_dropout_2': 0.3026873607952611, 'gat_out_channels_3': 32, 'gat_heads_3': 4, 'gat_dropout_3': 0.27908908896116574, 'gat_out_channels_4': 32, 'gat_heads_4': 4, 'gat_dropout_4': 0.09009827011104267, 'gat_out_channels_5': 32, 'gat_heads_5': 8, 'gat_dropout_5': 0.29601955931346974, 'gat_out_channels_6': 128, 'gat_heads_6': 1, 'gat_dropout_6': 0.4223143192539468, 'pooling_type': 'max', 'num_linear_layers': 3, 'linear_out_features_0

In [ ]:
import logging
import os
import pickle
import sys
from pathlib import Path
from typing import Optional, Tuple
import defopt
import grid2op
import matplotlib.pyplot as plt
import numpy as np
from grid2op.Agent import DoNothingAgent
from grid2op.Environment import BaseEnv
from lightsim2grid import LightSimBackend
from grid2op.Observation import CompleteObservation

from evaluation.score_agent import load_or_run, render_report
from evaluation.general_tutor import GeneralTutor
HOME_PATH =Path("../seeds_results/00/soft_seeds_ai4realnet_small")


In [7]:
# Paths
env_path = Path("../data/validation_envs") /"20_seeds_ai4realnet_small/"
ppath = Path(".")
actions_list = ppath / "actions" / "soft_actions.npy"
#topo_path = Path(DATA_PATH) / "junior"/"wcci2022_topo/"


##############
# Environment
##############
# Note: In order for this to work, you have to duplicate your validation environment 
# by the number of seeds you want to run. This needs to be done to ensure that the 
# the DoNothing Stastistics are independend from each other 
backend = LightSimBackend()
env = grid2op.make(
    env_path  / f"ai4realnet_small_1016",
    backend=LightSimBackend(), observation_class=CompleteObservation)
env.generate_classes()
#env = grid2op.make(
#    env_path  / f"l2rpn_2022_val_{seed}",
#    backend=LightSimBackend(), experimental_read_from_local_dir=True)


/mnt/stud/home/mhasan/miniconda3/envs/grid/lib/python3.12/site-packages/lightsim2grid/gridmodel/from_pandapower/_aux_add_trafo.py:79: UserWarning: There were some Nan in the pp_net.trafo["tap_step_degree"], they have been replaced by 0
  warnings.warn("There were some Nan in the pp_net.trafo[\"tap_step_degree\"], they have been replaced by 0")
/mnt/stud/home/mhasan/miniconda3/envs/grid/lib/python3.12/site-packages/lightsim2grid/gridmodel/from_pandapower/_aux_add_slack.py:114: UserWarning: We found either some slack coefficient to be < 0. or they were all 0.We set them all to 1.0 to avoid such issues
  warnings.warn("We found either some slack coefficient to be < 0. or they were all 0."


In [8]:
number_of_runs = len(os.listdir(env.chronics_handler.path))

In [9]:
print(number_of_runs)

24
